<a href="https://colab.research.google.com/github/jabri62018/Zx_Mother_Function_Jabri/blob/Zx_Mother_Function_Jabri/Zx_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

!pip install mpmath -q
import mpmath as mp
mp.mp.dps = 40

def calc_Cn(gamma):
    s = 0.5 + 1j*mp.mpf(gamma)
    try:
        zeta = mp.zeta(s)
        if abs(zeta) < 1e-20:
            return np.nan
        zp = mp.diff(mp.zeta, s)
        zpp = mp.diff(lambda z: mp.diff(mp.zeta, z), s)
        val = 0.5 * gamma**2 * mp.re(zpp/zeta)
        return float(val) if mp.isfinite(val) else np.nan
    except:
        return np.nan

print("="*80)
print(" ZX MODEL - ZERO INPUT - 28 CONSTANTS ".center(80))
print("="*80)

# 28 صفر أول من أصفار ريمان
roots = [
    14.1347251417347, 21.0220396387715, 25.0108575801459, 30.4248761258595,
    32.9350615877392, 37.5861781588257, 40.9187190121475, 43.3270732805630,
    48.0051508811672, 49.7738324778921, 52.9703214777146, 56.4462476970634,
    59.3479261202333, 60.8317785343542, 65.1125440480816, 67.0798105294949,
    69.5464017112022, 72.0671576744819, 75.7046906990068, 77.1448400704725,
    79.3373750202495, 82.9103808542536, 84.7354929800906, 87.4252746132682,
    88.8091112084165, 92.4918992713739, 94.6513440400545, 95.8706342280193
]

print(f"\n[1/5] Loaded {len(roots)} Riemann zeros")

print("\n[2/5] Calculating C_n constants...")
constants = []
for i, r in enumerate(roots):
    c = calc_Cn(r)
    if not np.isnan(c) and np.isfinite(c):
        constants.append(c)
        if (i+1) % 5 == 0:
            print(f" Calculated {i+1}/28 zeros...")
    time.sleep(0.05)

print(f" Successfully computed {len(constants)} constants")

if len(constants) < 10:
    raise RuntimeError(f"Only got {len(constants)} constants. Need at least 10.")

print("\n[3/5] Scaling to physical constants...")
# أول 28 ثابت فيزيائي للتجربة
const_names = [
    "c","G","hbar","h","e","me","mp","mu","eps0","alpha",
    "R_inf","kB","NA","R_gas","F","sigma","b","b_prime",
    "lambda_C","r_e","a0","mu_B","mu_N","g_e","g_p","g_n","Ry","Eh"
]
phys_vals = np.array([
    299792458.0, 6.674e-11, 1.054571817e-34, 6.62607015e-34,
    1.602176634e-19, 9.1093837015e-31, 1.67262192369e-27, 1.25663706212e-6,
    8.854187817e-12, 7.2973525693e-3, 10973731.568160, 1.380649e-23,
    6.02214076e23, 8.314462618, 96485.33212, 5.670374419e-8,
    2.897771955e-3, 2.8977719e-3, 2.42631023867e-12, 2.8179403227e-15,
    5.29177210903e-11, 927.40100783e-26, 5.0507837461e-27, 2.00231930436256,
    5.5856946893, -3.82608545, 10973731.6, 4.3597447222071e-18
])

n_const = min(28, len(constants))
C_abs = np.abs(np.array(constants[:n_const]))
S = phys_vals[1] / C_abs[1] # سكيل على G
C_scaled = C_abs * S

print("\n[4/5] Comparing ZX vs Physics...")
print("="*80)
matched_25 = 0
matched_10 = 0
for i in range(n_const):
    err = abs(C_scaled[i] - phys_vals[i]) / phys_vals[i]
    if err < 0.25:
        matched_25 += 1
    if err < 0.10:
        matched_10 += 1
    status = "OK" if err < 0.25 else "FAIL"
    print(f"{i+1:2d}. {const_names[i]:<8} | ZX={C_scaled[i]:.4e} | Phys={phys_vals[i]:.4e} | Err={err:6.2%} | {status}")

print("="*80)
print(f" Matched within 25%: {matched_25}/{n_const}")
print(f" Matched within 10%: {matched_10}/{n_const}")
print("="*80)

print("\n[5/5] Displaying results...")
df = pd.DataFrame({
    "n": range(1, n_const+1),
    "Constant": const_names[:n_const],
    "ZX_Scaled": C_scaled,
    "Physics": phys_vals[:n_const],
    "Rel_Error": [abs(C_scaled[i]-phys_vals[i])/phys_vals[i] for i in range(n_const)]
})

print("\n--- Results Table ---")
print(df.to_string(index=False, float_format="%.4e"))

plt.figure(figsize=(12,6))
plt.semilogy(range(1,len(constants)+1), np.abs(constants), 'ro-', alpha=0.7)
plt.xlabel("n"); plt.ylabel("|C_n|")
plt.title("ZX Constants from Riemann Zeros"); plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(12,6))
plt.loglog(range(1,n_const+1), C_scaled, 'bo-', label='ZX Model', alpha=0.8)
plt.loglog(range(1,n_const+1), phys_vals[:n_const], 'r--', label='Physics', alpha=0.8)
plt.xlabel("n"); plt.ylabel("Value")
plt.legend(); plt.title("ZX Model vs Physics Constants"); plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(12,5))
errors = [abs(C_scaled[i]-phys_vals[i])/phys_vals[i]*100 for i in range(n_const)]
plt.bar(range(1,n_const+1), errors)
plt.axhline(y=25, color='r', linestyle='--', label='25% threshold')
plt.xlabel("Constant n"); plt.ylabel("Error %")
plt.title("Relative Error per Constant"); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

print("\n" + "="*80)
print(" DONE - All results displayed on screen. Zero files saved. ".center(80))
print("="*80)

                      ZX MODEL - ZERO INPUT - 28 CONSTANTS                      

[1/5] Loaded 28 Riemann zeros

[2/5] Calculating C_n constants...
 Calculated 5/28 zeros...
 Calculated 10/28 zeros...
 Calculated 15/28 zeros...
 Calculated 20/28 zeros...
 Calculated 25/28 zeros...
 Successfully computed 28 constants

[3/5] Scaling to physical constants...

[4/5] Comparing ZX vs Physics...
 1. c        | ZX=4.2492e-10 | Phys=2.9979e+08 | Err=100.00% | FAIL
 2. G        | ZX=6.6740e-11 | Phys=6.6740e-11 | Err= 0.00% | OK
 3. hbar     | ZX=6.1606e-11 | Phys=1.0546e-34 | Err=58418258603852951285923840.00% | FAIL
 4. h        | ZX=2.9329e-09 | Phys=6.6261e-34 | Err=442625521980176699500789760.00% | FAIL
 5. e        | ZX=5.0831e-09 | Phys=1.6022e-19 | Err=3172644553701.12% | FAIL
 6. me       | ZX=7.0682e-10 | Phys=9.1094e-31 | Err=77592668999810012938240.00% | FAIL
 7. mp       | ZX=9.5463e-09 | Phys=1.6726e-27 | Err=570740100736636813312.00% | FAIL
 8. mu       | ZX=2.5185e-13 | Phys=1.25